## Step 1: Setup

In [1]:
#=== HomeGuard Security System ===
# Client: The Peterson Family
# Author: Gordan
# Description: The HomeGuard system processes sensor readings from a
# vacation home and routes them into three categories:
#   - Security Alerts   (intrusion-related: motion/door while AWAY)
#   - Safety Alerts      (frozen pipe / equipment failure / fire risk)
#   - Comfort Notifications (temperature outside the comfort band while HOME)
#======================================================================

import random
from datetime import datetime


## Step 2: Data Structures (Dictionaries)

Before wrapping sensors in a class, we represent a sensor reading and an
alert as plain dictionaries. The alert log (used for the end-of-run
summary) is built entirely out of these dictionaries.

In [2]:
def create_sensor_dict(sensor_id, location, sensor_type):
    """Returns a dictionary describing one sensor (id, location, type, value)."""
    return {
        "id": sensor_id,
        "location": location,
        "type": sensor_type,
        "current_value": None,
    }


def create_alert_dict(severity, category, message, location, timestamp):
    """Returns a dictionary describing one triggered alert, used to build the alert log."""
    return {
        "severity": severity,      # "HIGH", "MEDIUM", or "LOW"
        "category": category,      # "SECURITY", "SAFETY", or "COMFORT"
        "message": message,
        "location": location,
        "timestamp": timestamp,
    }


# Quick demo of the dictionary structures (Step 2 checkpoint)
demo_sensor = create_sensor_dict("S2", "Front Door", "door")
demo_sensor["current_value"] = "OPENED"
demo_alert = create_alert_dict(
    "HIGH", "SECURITY", "Front Door opened while in AWAY mode!",
    demo_sensor["location"], datetime.now().strftime("%H:%M:%S")
)
print("Sensor dict:", demo_sensor)
print("Alert dict:", demo_alert)


Sensor dict: {'id': 'S2', 'location': 'Front Door', 'type': 'door', 'current_value': 'OPENED'}
Alert dict: {'severity': 'HIGH', 'category': 'SECURITY', 'message': 'Front Door opened while in AWAY mode!', 'location': 'Front Door', 'timestamp': '23:35:39'}


## Step 3 + Step 5: Threshold Logic and the `Sensor` Class

The `Sensor` class combines the sensor's data with the behavior needed to
read and evaluate it. Two thresholds matter for temperature, and they are
kept separate on purpose (this was the bug flagged in the previous
submission â€” SAFETY and COMFORT were collapsed into a single range):

- **SAFETY** (checked in every mode): below 35Â°F is a frozen-pipe risk,
  above 95Â°F is an equipment-failure / fire risk.
- **COMFORT** (checked only in HOME mode): outside 65-75Â°F is merely a
  comfort notification, not a safety issue.

In [3]:
class Sensor:
    # Safety thresholds apply in every system mode.
    SAFETY_LOW = 35
    SAFETY_HIGH = 95
    # Comfort band only matters while someone is HOME.
    COMFORT_LOW = 65
    COMFORT_HIGH = 75

    def __init__(self, sensor_id, location, sensor_type):
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.current_value = None

    def read(self, forced_value=None):
        """Generates a new reading. Pass forced_value for deterministic test cases."""
        if forced_value is not None:
            self.current_value = forced_value
            return self.current_value

        if self.type == "temperature":
            self.current_value = random.randint(20, 105)
        elif self.type == "door":
            self.current_value = random.choice(["CLOSED", "OPENED"])
        elif self.type == "smoke":
            self.current_value = random.choices(["CLEAR", "DETECTED"], weights=[95, 5])[0]
        elif self.type == "motion":
            self.current_value = random.choices(["No activity", "DETECTED"], weights=[90, 10])[0]
        return self.current_value

    def isAbnormal(self):
        """SAFETY check only: frozen pipe / equipment failure / fire / intrusion signal."""
        if self.type == "temperature":
            return self.current_value < self.SAFETY_LOW or self.current_value > self.SAFETY_HIGH
        elif self.type == "door":
            return self.current_value == "OPENED"
        elif self.type == "smoke":
            return self.current_value == "DETECTED"
        elif self.type == "motion":
            return self.current_value == "DETECTED"
        return False

    def isUncomfortable(self):
        """COMFORT check: only applies to temperature sensors, independent of SAFETY."""
        if self.type != "temperature":
            return False
        return self.current_value < self.COMFORT_LOW or self.current_value > self.COMFORT_HIGH

    def reset(self):
        """Resets the sensor's value to None (used when re-arming the system)."""
        self.current_value = None


# Checkpoint: isAbnormal() vs isUncomfortable() are independent checks
test_sensor = Sensor("S3", "Kitchen", "temperature")
for value in [20, 68, 80, 105]:
    test_sensor.read(forced_value=value)
    print(f"{value}F -> isAbnormal(SAFETY)={test_sensor.isAbnormal()}  "
          f"isUncomfortable(COMFORT)={test_sensor.isUncomfortable()}")


20F -> isAbnormal(SAFETY)=True  isUncomfortable(COMFORT)=True
68F -> isAbnormal(SAFETY)=False  isUncomfortable(COMFORT)=False
80F -> isAbnormal(SAFETY)=False  isUncomfortable(COMFORT)=True
105F -> isAbnormal(SAFETY)=True  isUncomfortable(COMFORT)=True


## Step 4: Functions (Logging + Alert Routing)

`trigger_alert` routes a sensor's current reading to the correct
category/severity. It also detects one extra edge case: a door AND
motion sensor both triggering in the *same* reading cycle while AWAY is
escalated to a "possible break-in" alert.

In [4]:
def log_event(message, alerts_log=None, severity=None, category=None, location=None):
    """Prints a timestamped log line and, if a log list is supplied, appends an alert dict to it."""
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")
    if alerts_log is not None and severity is not None:
        alerts_log.append(create_alert_dict(severity, category, message, location, timestamp))


def trigger_alert(sensor, system_mode, alerts_log):
    """
    Decides, based on sensor type/value and system mode, which alert (if any)
    to raise. Every raised alert is also recorded as a dictionary in alerts_log.
    """
    if sensor.type == "door" and sensor.current_value == "OPENED" and system_mode == "AWAY":
        msg = "SECURITY: Front Door opened while in AWAY mode!"
        print(f"[ALERT!] \U0001F6A8 HIGH: {msg}")
        log_event("Sending notification to homeowner...", alerts_log, "HIGH", "SECURITY", sensor.location)

    elif sensor.type == "smoke" and sensor.current_value == "DETECTED":
        msg = f"SAFETY: Smoke detected in {sensor.location}!"
        print(f"[ALERT!] \U0001F6A8 HIGH: {msg}")
        log_event("Sending notification to homeowner...", alerts_log, "HIGH", "SAFETY", sensor.location)

    elif sensor.type == "temperature" and sensor.isAbnormal():
        if sensor.current_value < Sensor.SAFETY_LOW:
            msg = f"SAFETY: {sensor.location} temperature {sensor.current_value}F - frozen pipe risk!"
        else:
            msg = f"SAFETY: {sensor.location} temperature {sensor.current_value}F - equipment failure/fire risk!"
        print(f"[ALERT!] \U0001F6A8 HIGH: {msg}")
        log_event("Sending notification to homeowner...", alerts_log, "HIGH", "SAFETY", sensor.location)

    elif sensor.type == "motion" and sensor.current_value == "DETECTED" and system_mode == "AWAY":
        msg = f"SECURITY: Motion detected in {sensor.location}!"
        print(f"[ALERT!] \u26A0\uFE0F MEDIUM: {msg}")
        log_event("Sending notification to homeowner...", alerts_log, "MEDIUM", "SECURITY", sensor.location)

    elif sensor.type == "temperature" and system_mode == "HOME" and sensor.isUncomfortable():
        msg = f"COMFORT: {sensor.location} temperature {sensor.current_value}F is outside the comfort range."
        print(f"[NOTICE] \U0001F4CB LOW: {msg}")


## Step 6: Main Simulation Loop

`run_simulation` accepts an optional `forced_readings` dict so we can
run **fixed, deterministic test cases** (as the lab's pseudocode asks
for) instead of relying on random luck to hit a threshold. It also
implements the "multiple sensors triggered simultaneously" edge case:
if both the door and a motion sensor trip in the same cycle while AWAY,
it escalates to a break-in alert.

In [5]:
def run_simulation(system_mode="AWAY", iterations=3, forced_readings=None):
    """
    Creates the sensor list and runs the simulation for a set number of
    time steps, printing readings and any triggered alerts.

    forced_readings: optional dict {sensor_id: [value_iter0, value_iter1, ...]}
    used to build deterministic, reproducible test cases.
    """
    forced_readings = forced_readings or {}

    sensor_list = [
        Sensor("S1", "Living Room", "motion"),
        Sensor("S2", "Front Door", "door"),
        Sensor("S3", "Kitchen", "temperature"),
        Sensor("S4", "Bedroom", "smoke"),
    ]

    alerts_log = []

    print("=== HomeGuard Security System ===")
    print(f"Time: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Mode: {system_mode}\n")

    for i in range(iterations):
        if i > 0:
            print(f"\nTime: {datetime.now().strftime('%H:%M:%S')}")

        triggered_types = set()

        for sensor in sensor_list:
            forced_sequence = forced_readings.get(sensor.id)
            forced_value = forced_sequence[i] if forced_sequence and i < len(forced_sequence) else None
            sensor.read(forced_value=forced_value)
            print(f"[READING] {sensor.location} {sensor.type.capitalize()}: {sensor.current_value}")

            if sensor.isAbnormal():
                trigger_alert(sensor, system_mode, alerts_log)
                triggered_types.add(sensor.type)
            elif sensor.isUncomfortable():
                trigger_alert(sensor, system_mode, alerts_log)

        # Edge case: door + motion both tripped in the same cycle while AWAY
        # is treated as a possible break-in and escalated above the two
        # individual alerts already raised above.
        if system_mode == "AWAY" and "door" in triggered_types and "motion" in triggered_types:
            msg = "SECURITY: Door AND motion triggered in the same cycle - possible break-in!"
            print(f"[ALERT!] \U0001F6A8 HIGH: {msg}")
            log_event("Escalating to emergency contact...", alerts_log, "HIGH", "SECURITY", "Whole House")

    # End-of-run summary built from the alert-dict log (Step 2 dictionaries in action)
    print("\n--- Alert Summary ---")
    if not alerts_log:
        print("No alerts were triggered this run.")
    else:
        by_severity = {}
        for alert in alerts_log:
            by_severity[alert["severity"]] = by_severity.get(alert["severity"], 0) + 1
        for severity, count in by_severity.items():
            print(f"{severity}: {count}")

    return alerts_log


# ---------------------------------------------------------
# Entry point
# ---------------------------------------------------------
if __name__ == "__main__":
    run_simulation(system_mode="HOME", iterations=3)


## Step 7: Test Scenarios (fixed, deterministic cases)

Three separate runs, each forcing specific sensor values so the output is
reproducible and each requirement in "System Requirements" is
demonstrably exercised.

### Test 1 - Security (AWAY mode): door + motion, including simultaneous trigger

In [6]:
security_forced = {
    "S1": ["No activity", "DETECTED", "DETECTED"],   # motion: quiet, then detected twice
    "S2": ["CLOSED", "OPENED", "OPENED"],             # door: quiet, then opened (matches motion on iter 2)
    "S3": [70, 70, 70],                                # temperature: stays safe/comfortable, not the focus here
    "S4": ["CLEAR", "CLEAR", "CLEAR"],
}
security_alerts = run_simulation(system_mode="AWAY", iterations=3, forced_readings=security_forced)


=== HomeGuard Security System ===
Time: 23:35:39
Mode: AWAY

[READING] Living Room Motion: No activity
[READING] Front Door Door: CLOSED
[READING] Kitchen Temperature: 70
[READING] Bedroom Smoke: CLEAR

Time: 23:35:39
[READING] Living Room Motion: DETECTED
[ALERT!] ⚠️ MEDIUM: SECURITY: Motion detected in Living Room!
[LOG] [23:35:39] Sending notification to homeowner...
[READING] Front Door Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [23:35:39] Sending notification to homeowner...
[READING] Kitchen Temperature: 70
[READING] Bedroom Smoke: CLEAR
[ALERT!] 🚨 HIGH: SECURITY: Door AND motion triggered in the same cycle - possible break-in!
[LOG] [23:35:39] Escalating to emergency contact...

Time: 23:35:39
[READING] Living Room Motion: DETECTED
[ALERT!] ⚠️ MEDIUM: SECURITY: Motion detected in Living Room!
[LOG] [23:35:39] Sending notification to homeowner...
[READING] Front Door Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY m

### Test 2 - Safety: frozen pipe, equipment failure, and smoke

In [7]:
safety_forced = {
    "S1": ["No activity", "No activity", "No activity"],
    "S2": ["CLOSED", "CLOSED", "CLOSED"],
    "S3": [20, 105, 70],       # iter0: frozen pipe risk, iter1: equipment failure/fire risk, iter2: safe
    "S4": ["CLEAR", "CLEAR", "DETECTED"],   # smoke detected on iter2
}
safety_alerts = run_simulation(system_mode="HOME", iterations=3, forced_readings=safety_forced)


=== HomeGuard Security System ===
Time: 23:35:39
Mode: HOME

[READING] Living Room Motion: No activity
[READING] Front Door Door: CLOSED
[READING] Kitchen Temperature: 20
[ALERT!] 🚨 HIGH: SAFETY: Kitchen temperature 20F - frozen pipe risk!
[LOG] [23:35:39] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Time: 23:35:39
[READING] Living Room Motion: No activity
[READING] Front Door Door: CLOSED
[READING] Kitchen Temperature: 105
[ALERT!] 🚨 HIGH: SAFETY: Kitchen temperature 105F - equipment failure/fire risk!
[LOG] [23:35:39] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Time: 23:35:39
[READING] Living Room Motion: No activity
[READING] Front Door Door: CLOSED
[READING] Kitchen Temperature: 70
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🚨 HIGH: SAFETY: Smoke detected in Bedroom!
[LOG] [23:35:39] Sending notification to homeowner...

--- Alert Summary ---
HIGH: 3


### Test 3 - Comfort (HOME mode): temperature outside 65-75F but inside safety range

In [8]:
comfort_forced = {
    "S1": ["No activity", "No activity", "No activity"],
    "S2": ["CLOSED", "CLOSED", "CLOSED"],
    "S3": [80, 60, 70],   # iter0: too warm (comfort only), iter1: too cool (comfort only), iter2: comfortable
    "S4": ["CLEAR", "CLEAR", "CLEAR"],
}
comfort_alerts = run_simulation(system_mode="HOME", iterations=3, forced_readings=comfort_forced)


=== HomeGuard Security System ===
Time: 23:35:39
Mode: HOME

[READING] Living Room Motion: No activity
[READING] Front Door Door: CLOSED
[READING] Kitchen Temperature: 80
[NOTICE] 📋 LOW: COMFORT: Kitchen temperature 80F is outside the comfort range.
[READING] Bedroom Smoke: CLEAR

Time: 23:35:39
[READING] Living Room Motion: No activity
[READING] Front Door Door: CLOSED
[READING] Kitchen Temperature: 60
[NOTICE] 📋 LOW: COMFORT: Kitchen temperature 60F is outside the comfort range.
[READING] Bedroom Smoke: CLEAR

Time: 23:35:39
[READING] Living Room Motion: No activity
[READING] Front Door Door: CLOSED
[READING] Kitchen Temperature: 70
[READING] Bedroom Smoke: CLEAR

--- Alert Summary ---
No alerts were triggered this run.
